# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/si-ux/FlyrankAI-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

> **Lane:** Ranking Signal Analysis · **Card:** ML-08 · Same frame, same label, same metric as
> ML-07 (`w04_baseline_score.ipynb`), so the comparison is apples to apples.
>
> Seeds are fixed (`SEED = 20260808`). Tree ensembles move a point or two across library
> versions; the versions used are printed below.

In [1]:
%pip install -q pandas scikit-learn pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json, pathlib, subprocess, sys, time
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 20260808

REPO = pathlib.Path.cwd()
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (p / "data/raw/content_refresh_anonymized.csv").exists():
        REPO = p; break
OUT = REPO / "work/outputs"; OUT.mkdir(parents=True, exist_ok=True)
rel = lambda q: q.relative_to(REPO).as_posix()

FRAME = OUT / "modeling_frame_dev.parquet"
if not FRAME.exists():
    subprocess.run([sys.executable, str(REPO / "work/scripts/build_modeling_frame.py")], check=True)

df     = pd.read_parquet(FRAME).reset_index(drop=True)
y      = df["is_position_decline"].values
groups = df["client_hash_id"].values
BASE_RATE = y.mean()

print(f"pandas {pd.__version__} · numpy {np.__version__} · scikit-learn {sklearn.__version__}")
print(f"{len(df):,} content items · {df.client_hash_id.nunique()} clients · "
      f"base rate {BASE_RATE:.4f}")

pandas 3.0.3 · numpy 2.5.1 · scikit-learn 1.9.0
106,461 content items · 42 clients · base rate 0.5673


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**The question is "which pages should a human look at first?"** — a *ranking* question, not a
yes/no one. Nobody acts on 106,461 individual verdicts; they work down a list until the day ends.

So I train **classifiers but read their probabilities as a ranking**, and evaluate at
**precision@K** — the same metric the ML-07 baseline was scored on. Three methods, in
increasing order of opacity, because simplicity is a feature:

| Method | Why it is here |
|---|---|
| **Logistic regression** | the readable floor: monotone in each feature, coefficients you can state in a sentence |
| **Decision tree (depth 4)** | printable — the whole rule fits on a screen and can be checked by eye against the ML-07 hand rule |
| **Random forest** | the stronger model; earns its opacity only if it actually beats the two above |

I am not reaching for gradient boosting or a neural net. The honest comparison below shows the
learned models barely clear a hand-written rule on this problem — extra capacity is not the
missing ingredient, and an opaque model that wins by 0.02 would be a worse deliverable.

In [3]:
NUM = ["f_pos","f_impressions","f_clicks","f_ctr","f_days_with_impressions","f_pos_volatility",
       "f_pos_trend","search_volume","competition","cpc","backlinks","word_count","char_count",
       "keyword_token_count","content_age_days"]
CAT = ["content_type","main_intent","competition_level"]

# Missingness follows content_type, so a blind median-fill would smuggle content type into the
# features. Flag it explicitly instead, then impute.
FLAG_COLS = ["f_pos_trend","backlinks","word_count","search_volume"]

def make_X(d, client_relative=False):
    X = d[NUM + CAT].copy()
    for c in FLAG_COLS:
        X[f"has_{c}"] = X[c].notna().astype(int)
    if client_relative:
        g = d.groupby("client_hash_id")
        X["cr_pos"]   = d.f_pos - g.f_pos.transform("median")
        X["cr_trend"] = d.f_pos_trend.fillna(0) - g.f_pos_trend.transform(lambda s: s.fillna(0).median())
        X["cr_imps"]  = np.log1p(d.f_impressions) - g.f_impressions.transform(lambda s: np.log1p(s).median())
    return X, [c for c in X.columns if c not in CAT]

def pipe_for(est, num):
    return Pipeline([("pre", ColumnTransformer([
        ("num", Pipeline([("i", SimpleImputer(strategy="median")), ("s", StandardScaler())]), num),
        ("cat", Pipeline([("i", SimpleImputer(strategy="constant", fill_value="unknown")),
                          ("o", OneHotEncoder(handle_unknown="ignore", min_frequency=50))]), CAT),
    ])), ("clf", est)])

MODELS = {
    "logistic_regression": lambda: LogisticRegression(max_iter=2000, random_state=SEED),
    "decision_tree_d4":    lambda: DecisionTreeClassifier(max_depth=4, min_samples_leaf=200,
                                                          random_state=SEED),
    "random_forest":       lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                          n_jobs=-1, random_state=SEED),
}
X_all, NUMF = make_X(df)
print(f"{len(NUMF)} numeric + {len(CAT)} categorical features "
      f"({len(FLAG_COLS)} explicit missingness flags)")

19 numeric + 3 categorical features (4 explicit missingness flags)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by client, 5-fold — and the split is the single most consequential choice here.**

Pages are not independent. They come in client portfolios that share a CMS, a template, a
publishing cadence, and one SEO team's habits. A random split puts a client's pages on **both**
sides of the line, so the model can recognise the client and recite that client's average
outcome instead of learning what precedes a decline.

The honest question is *"does this work on a client it has never seen?"* — which is exactly the
deployment case, because a new client arrives with no history in the training set.

So: `GroupKFold(n_splits=5)` on `client_hash_id`. Every page gets an **out-of-fold** prediction
from a model that never saw its client. All numbers below are OOF — never in-sample.

I also run the random split, not to report it as my result, but because **the gap between the
two is itself a finding** about how much memorisation is available.

In [4]:
def oof_predict(splitter, split_kw, models, X, num, tag):
    """Out-of-fold probabilities; cached so reruns are cheap."""
    preds = {}
    for name, make in models.items():
        f = OUT / f"w05_oof_{tag}_{name}.npy"
        if f.exists():
            preds[name] = np.load(f); continue
        t0 = time.time(); p = np.zeros(len(X))
        for tr, te in splitter.split(X, y, **split_kw):
            p[te] = pipe_for(make(), num).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]
        np.save(f, p); preds[name] = p
        print(f"  {tag:9s} {name:20s} {time.time()-t0:5.0f}s")
    return preds

grouped = oof_predict(GroupKFold(n_splits=5), {"groups": groups}, MODELS, X_all, NUMF, "grouped")
random_ = oof_predict(StratifiedKFold(5, shuffle=True, random_state=SEED), {}, MODELS, X_all, NUMF, "random")

# fold hygiene: no client may appear in both sides of any split
for tr, te in GroupKFold(n_splits=5).split(X_all, y, groups=groups):
    assert not (set(groups[tr]) & set(groups[te])), "client leaked across the grouped split"
print("\nGrouped split verified: no client appears in both train and test of any fold.")
sizes = [len(te) for _, te in GroupKFold(n_splits=5).split(X_all, y, groups=groups)]
print("fold test sizes:", sizes)

  grouped   logistic_regression      8s


  grouped   decision_tree_d4         7s


  grouped   random_forest           96s


  random    logistic_regression      8s


  random    decision_tree_d4         7s


  random    random_forest           97s



Grouped split verified: no client appears in both train and test of any fold.
fold test sizes: [23400, 20766, 20765, 20765, 20765]


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# The ML-07 baseline, recomputed here on the identical rows so the table is one run.
vol_median = df["f_pos_volatility"].median()
trend      = df["f_pos_trend"].fillna(0.0)
pts = ((trend >= 1.0).astype(int) * 3
       + (df["f_pos_volatility"].fillna(0) >= vol_median).astype(int)
       + ((df["f_pos"] > 0) & (df["f_pos"] <= 20)).astype(int)
       + (df["f_impressions"] >= 1000).astype(int)
       + (df["f_days_with_impressions"] < 60).astype(int)).values
expo = np.log1p(df["f_impressions"].values); expo = (expo - expo.min()) / (expo.max() - expo.min())
baseline_score = pts + 0.999 * expo

KS = [50, 100, 500, 1000, 5000]
def patk(s, k):
    return y[np.argsort(-np.asarray(s), kind="stable")[:k]].mean()

def row(name, s):
    return {"model": name, "roc_auc": roc_auc_score(y, s),
            "avg_precision": average_precision_score(y, s),
            **{f"P@{k}": patk(s, k) for k in KS},
            "distinct_scores": len(np.unique(np.round(s, 6)))}

table = [row("base rate (random pick)", np.random.RandomState(SEED).rand(len(y))),
         row("baseline_rule (ML-07)", baseline_score)]
table += [row(f"{n} [grouped]", p) for n, p in grouped.items()]
res = pd.DataFrame(table).set_index("model")
print("HONEST SPLIT — grouped by client, out-of-fold\n")
print(res.round(3).to_string())
print(f"\nbase rate = {BASE_RATE:.4f}: any P@K below this is worse than picking at random.")

HONEST SPLIT — grouped by client, out-of-fold

                               roc_auc  avg_precision  P@50  P@100  P@500  P@1000  P@5000  distinct_scores
model                                                                                                     
base rate (random pick)          0.499          0.567  0.66   0.62  0.580   0.559   0.569           101024
baseline_rule (ML-07)            0.635          0.696  0.76   0.76  0.846   0.847   0.842            39743
logistic_regression [grouped]    0.617          0.666  0.74   0.75  0.682   0.690   0.751            95126
decision_tree_d4 [grouped]       0.612          0.658  0.32   0.33  0.464   0.478   0.694               78
random_forest [grouped]          0.649          0.701  0.80   0.83  0.832   0.846   0.829            97495

base rate = 0.5673: any P@K below this is worse than picking at random.


In [6]:
# The same models under a random split — reported as a diagnostic, not as a result.
comp = pd.DataFrame([row(f"{n} [random]", p) for n, p in random_.items()]).set_index("model")
print("OPTIMISTIC SPLIT — random, clients on both sides (NOT my result)\n")
print(comp.round(3).to_string())

gap = pd.DataFrame({
    "grouped_auc": [roc_auc_score(y, grouped[n]) for n in MODELS],
    "random_auc":  [roc_auc_score(y, random_[n]) for n in MODELS],
    "grouped_P@50": [patk(grouped[n], 50) for n in MODELS],
    "random_P@50":  [patk(random_[n], 50) for n in MODELS],
}, index=list(MODELS))
gap["auc_gap"] = gap.random_auc - gap.grouped_auc
print("\n\nThe gap IS the finding:\n")
print(gap.round(3).to_string())

OPTIMISTIC SPLIT — random, clients on both sides (NOT my result)

                              roc_auc  avg_precision  P@50  P@100  P@500  P@1000  P@5000  distinct_scores
model                                                                                                    
logistic_regression [random]    0.682          0.727  0.84   0.90  0.856   0.848   0.850            95483
decision_tree_d4 [random]       0.676          0.712  0.58   0.68  0.734   0.779   0.782               80
random_forest [random]          0.776          0.810  1.00   0.98  0.974   0.965   0.941            98941




The gap IS the finding:

                     grouped_auc  random_auc  grouped_P@50  random_P@50  auc_gap
logistic_regression        0.617       0.682          0.74         0.84    0.065
decision_tree_d4           0.612       0.676          0.32         0.58    0.065
random_forest              0.649       0.776          0.80         1.00    0.127


### What the table says

**The hand-written rule is genuinely hard to beat.** On the honest grouped split the random
forest reaches **P@50 = 0.800** against the baseline's **0.760**, and **AUC 0.649 vs 0.635**.
That is a real but modest win — roughly two extra correct pages in a top-50 review list. Any
write-up claiming the model "transformed" this problem would be overselling it.

**Logistic regression and the depth-4 tree do not beat the rule at all.** The tree's
`P@50 = 0.320` looks alarming until you read the `distinct_scores` column: a depth-4 tree has at
most 16 leaves, so 106,461 pages share **78** distinct probabilities. It cannot order a top-50 —
the same tie pathology that made the raw ML-07 rule score below base rate before I added the
exposure tie-break. It is a resolution failure, not a signal failure.

**The random split inflates everything, and that gap is the real lesson.** The random forest goes
from AUC 0.649 (grouped) to **0.776** (random), and from `P@50 = 0.800` to **1.000** — a perfect
top-50. Nothing about the model improved; it was simply allowed to see other pages belonging to
the same client and recite that client's average outcome. Had I reported the random-split number,
I would have claimed roughly **double** the true skill above base rate.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [7]:
from sklearn.inspection import permutation_importance

X_tr, num = make_X(df)
tr, te = next(iter(GroupKFold(n_splits=5).split(X_tr, y, groups=groups)))
rf1 = pipe_for(RandomForestClassifier(n_estimators=200, min_samples_leaf=5, n_jobs=-1,
                                      random_state=SEED), num).fit(X_tr.iloc[tr], y[tr])
sub = np.random.RandomState(SEED).choice(te, size=min(15000, len(te)), replace=False)
imp = permutation_importance(rf1, X_tr.iloc[sub], y[sub], n_repeats=5, random_state=SEED,
                             n_jobs=-1, scoring="roc_auc")
importance = pd.Series(imp.importances_mean, index=X_tr.columns).sort_values(ascending=False)
print("Permutation importance — AUC lost when the column is shuffled (grouped fold 1):\n")
print(importance.head(10).round(4).to_string())

Permutation importance — AUC lost when the column is shuffled (grouped fold 1):

f_pos_trend                0.1031
f_pos                      0.0317
f_pos_volatility           0.0199
f_days_with_impressions    0.0156
f_ctr                      0.0090
f_clicks                   0.0080
has_word_count             0.0032
char_count                 0.0025
keyword_token_count        0.0020
main_intent                0.0014


In [8]:
# f_pos_trend towers over everything. The skill says: a dominant feature is a leakage SYMPTOM.
# So run the prescribed test — train without it and see whether the score collapses.
f = OUT / "w05_oof_grouped_rf_no_trend.npy"
if f.exists():
    p_no_trend = np.load(f)
else:
    X_nt = X_all.drop(columns=["f_pos_trend", "has_f_pos_trend"])
    num_nt = [c for c in NUMF if c not in ("f_pos_trend", "has_f_pos_trend")]
    p_no_trend = np.zeros(len(df))
    for tr_, te_ in GroupKFold(n_splits=5).split(X_nt, y, groups=groups):
        p_no_trend[te_] = pipe_for(MODELS["random_forest"](), num_nt).fit(
            X_nt.iloc[tr_], y[tr_]).predict_proba(X_nt.iloc[te_])[:, 1]
    np.save(f, p_no_trend)

with_auc, without_auc = roc_auc_score(y, grouped["random_forest"]), roc_auc_score(y, p_no_trend)
print(f"random forest WITH  f_pos_trend : AUC {with_auc:.3f}  P@50 {patk(grouped['random_forest'],50):.3f}")
print(f"random forest WITHOUT           : AUC {without_auc:.3f}  P@50 {patk(p_no_trend,50):.3f}")
print(f"drop: {with_auc - without_auc:.3f} AUC")
print("\nLeakage verdict: NOT leakage. A label-derived feature collapses the score from ~1.0")
print("when removed; here the model was never near 1.0 (0.649) and degrades gracefully to")
print(f"{without_auc:.3f}. f_pos_trend is March position minus January position — both inside the")
print("feature window, both strictly before the April label. It dominates because prior")
print("direction genuinely is the strongest observable signal, which is also what ML-07 found.")

random forest WITH  f_pos_trend : AUC 0.649  P@50 0.800
random forest WITHOUT           : AUC 0.578  P@50 0.780
drop: 0.071 AUC

Leakage verdict: NOT leakage. A label-derived feature collapses the score from ~1.0
when removed; here the model was never near 1.0 (0.649) and degrades gracefully to
0.578. f_pos_trend is March position minus January position — both inside the
feature window, both strictly before the April label. It dominates because prior
direction genuinely is the strongest observable signal, which is also what ML-07 found.


In [9]:
d = df.assign(p=grouped["random_forest"])

print("Calibration — is a predicted 0.8 really 80%?\n")
d["decile"] = pd.qcut(d.p, 10, labels=False, duplicates="drop")
cal = d.groupby("decile").agg(n=("p","size"), mean_predicted=("p","mean"),
                              actual=("is_position_decline","mean"))
cal["error"] = cal.mean_predicted - cal.actual
print(cal.round(3).to_string())
print("\nThe model is over-confident at the top (predicts ~0.84, observes ~0.80) and")
print("under-confident at the bottom. Usable for RANKING; do not quote these as probabilities.")

print("\n\nPer-client AUC — does it work everywhere, or on average?\n")
per = (d.groupby("client_hash_id")
        .filter(lambda g: len(g) >= 300 and g.is_position_decline.nunique() > 1)
        .groupby("client_hash_id")
        .apply(lambda g: pd.Series({"pages": len(g),
                                    "decline_rate": g.is_position_decline.mean(),
                                    "auc": roc_auc_score(g.is_position_decline, g.p)}),
               include_groups=False)
        .sort_values("auc"))
per.index = ["c" + i[-4:] for i in per.index]      # short pseudonymous handles
print(per.round(3).to_string())
print(f"\nSpread: {per.auc.min():.3f} to {per.auc.max():.3f} across {len(per)} clients with "
      f">=300 pages.")

Calibration — is a predicted 0.8 really 80%?

            n  mean_predicted  actual  error
decile                                      
0       10647           0.277   0.363 -0.085
1       10646           0.399   0.425 -0.026
2       10646           0.461   0.465 -0.004
3       10646           0.509   0.507  0.003
4       10646           0.554   0.548  0.006
5       10646           0.597   0.586  0.011
6       10646           0.641   0.615  0.027
7       10646           0.692   0.658  0.034
8       10646           0.751   0.704  0.047
9       10646           0.835   0.803  0.032

The model is over-confident at the top (predicts ~0.84, observes ~0.80) and
under-confident at the bottom. Usable for RANKING; do not quote these as probabilities.


Per-client AUC — does it work everywhere, or on average?



         pages  decline_rate    auc
c7c86    482.0         0.143  0.480
c11d5   2032.0         0.770  0.526
c5515   1248.0         0.370  0.544
c1abf   1319.0         0.373  0.573
ca4a0    739.0         0.625  0.591
c46ef   1470.0         0.236  0.592
c81d4   4165.0         0.530  0.593
c4f3d   2316.0         0.645  0.597
cf01b    777.0         0.494  0.600
c8242   2931.0         0.594  0.608
ca37d    622.0         0.563  0.610
cb4db    948.0         0.879  0.617
cd1de   3007.0         0.396  0.620
c62c0   8834.0         0.204  0.625
c63c4  13025.0         0.630  0.630
c8636   8218.0         0.826  0.665
c65ea  23400.0         0.491  0.666
c7cbb   1064.0         0.849  0.690
c0096  18215.0         0.689  0.702
c3229   8816.0         0.562  0.722
cf715   1824.0         0.920  0.769

Spread: 0.480 to 0.769 across 21 clients with >=300 pages.


In [10]:
# Three concrete wrong cases, and what makes each hard.
d["page"] = ["p" + h[-6:] for h in d.content_hash_id]
cols = ["page","p","is_position_decline","f_pos","f_pos_trend","f_pos_volatility",
        "f_impressions","f_days_with_impressions","pos_delta"]

worst_fp = d[d.is_position_decline == 0].nlargest(3, "p")     # confident, and wrong
worst_fn = d[d.is_position_decline == 1].nsmallest(3, "p")    # dismissed, and wrong

print("Confidently predicted DECLINE, actually held or improved:\n")
print(worst_fp[cols].to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
print("\n\nConfidently predicted STABLE, actually declined:\n")
print(worst_fn[cols].to_string(index=False, float_format=lambda v: f"{v:,.2f}"))

Confidently predicted DECLINE, actually held or improved:

   page    p  is_position_decline  f_pos  f_pos_trend  f_pos_volatility  f_impressions  f_days_with_impressions  pos_delta
p082ef7 0.97                    0  13.80        16.39             13.95       2,111.00                       90      -3.02
pcc2777 0.95                    0  26.99        17.04             15.96       1,065.00                       90      -0.59
pc4082c 0.95                    0   7.37         8.01              7.00       1,960.00                       90       0.60


Confidently predicted STABLE, actually declined:

   page    p  is_position_decline  f_pos  f_pos_trend  f_pos_volatility  f_impressions  f_days_with_impressions  pos_delta
pc2e077 0.06                    1   2.80        -0.36              0.43      20,691.00                       90       1.36
p351a57 0.07                    1  65.63       -11.57             11.86         777.00                       89      11.95
p98f53a 0.08                

In [11]:
# Does addressing the ML-07 weak pick (one client owning the top of the queue) help?
f = OUT / "w05_oof_grouped_rf_client_relative.npy"
if f.exists():
    p_cr = np.load(f)
else:
    X_cr, num_cr = make_X(df, client_relative=True)
    p_cr = np.zeros(len(df))
    for tr_, te_ in GroupKFold(n_splits=5).split(X_cr, y, groups=groups):
        p_cr[te_] = pipe_for(MODELS["random_forest"](), num_cr).fit(
            X_cr.iloc[tr_], y[tr_]).predict_proba(X_cr.iloc[te_])[:, 1]
    np.save(f, p_cr)

final = pd.DataFrame([row("baseline_rule (ML-07)", baseline_score),
                      row("random_forest", grouped["random_forest"]),
                      row("random_forest + client-relative", p_cr)]).set_index("model")
print(final[["roc_auc","P@50","P@500","P@5000"]].round(3).to_string())

def top_conc(s, k=50):
    idx = np.argsort(-np.asarray(s), kind="stable")[:k]
    vc = df.client_hash_id.values[idx]
    return pd.Series(vc).value_counts().iloc[0] / k, pd.Series(vc).nunique()

for nm, s in [("baseline_rule", baseline_score), ("random_forest", grouped["random_forest"]),
              ("rf + client-relative", p_cr)]:
    share, nclients = top_conc(s)
    print(f"{nm:24s} top-50: largest client {share:.0%}, {nclients} distinct clients")

json.dump({
    "seed": SEED, "base_rate": round(float(BASE_RATE), 4),
    "split": "GroupKFold(5) on client_hash_id, out-of-fold",
    "sklearn": sklearn.__version__,
    "grouped": {n: {"roc_auc": round(float(roc_auc_score(y, p)), 4),
                    **{f"P@{k}": round(float(patk(p, k)), 4) for k in KS}}
                for n, p in {**grouped, "rf_client_relative": p_cr,
                             "rf_no_trend": p_no_trend}.items()},
    "random_split_diagnostic": {n: round(float(roc_auc_score(y, p)), 4) for n, p in random_.items()},
    "baseline_rule": {"roc_auc": round(float(roc_auc_score(y, baseline_score)), 4),
                      **{f"P@{k}": round(float(patk(baseline_score, k)), 4) for k in KS}},
}, open(OUT / "w05_model_metrics.json", "w"), indent=2)
print(f"\nreceipt -> {rel(OUT / 'w05_model_metrics.json')}")

                                 roc_auc  P@50  P@500  P@5000
model                                                        
baseline_rule (ML-07)              0.635  0.76  0.846   0.842
random_forest                      0.649  0.80  0.832   0.829
random_forest + client-relative    0.636  0.88  0.876   0.822
baseline_rule            top-50: largest client 66%, 8 distinct clients
random_forest            top-50: largest client 62%, 5 distinct clients
rf + client-relative     top-50: largest client 78%, 3 distinct clients



receipt -> work/outputs/w05_model_metrics.json


### Errors and interpretation — what I actually believe

**What the model leans on.** `f_pos_trend` (March position minus January position) dominates by
roughly 3× the next feature, followed by `f_pos`, `f_pos_volatility`, and
`f_days_with_impressions`. Every one of the top four is a *ranking-behaviour* signal; none of the
content-property features (`word_count`, `char_count`, `cpc`, `backlinks`) matters much. For a
lane called Ranking Signal Analysis, the observed answer is that **how a page has recently been
ranking predicts where it goes next far better than what the page is made of.**

**I checked the dominant feature for leakage and cleared it.** A feature that towers over the
rest is the classic leakage symptom, so I ran the prescribed test: retrain without it. A
label-derived feature collapses the score toward chance when removed; this one costs a modest
amount of AUC and the model degrades gracefully. `f_pos_trend` is computed entirely inside the
feature window, strictly before the label month. It is a strong signal, not a leak.

**Where it is wrong.** Per-client AUC ranges from about **0.48 to 0.77**. For at least one client
with hundreds of pages the model is no better than a coin flip. Reporting a single pooled AUC
hides that completely — the model does not "work"; it works *for some portfolios*. Any deployment
should report per-client performance and quietly abstain where it has none.

The concrete misses divide into two families. **Confident false positives** are pages with a
steep prior slide that then stabilised — the model cannot tell genuine decay from a page settling
back after an unusually good January, which is exactly the regression-to-the-mean ambiguity
flagged in ML-07. **Confident false negatives** are pages that looked stable for ninety days and
dropped anyway, driven by something the panel cannot observe: a competitor's move, a SERP layout
change, an algorithm update. No feature in this contract can anticipate those.

**Calibration:** monotone but over-confident at the top. Fine for ordering a queue, not fine for
quoting "this page has an 84% chance of declining". I use these as ranks, not probabilities.

**The client-relative variant improves the ranking but does NOT fix ML-07's weak pick.**
Adding three features that compare a page to its *own client's* median position, trend, and
traffic lifts precision@50 from 0.800 to **0.880** — the best top-of-queue number in this
project — at a small cost in pooled AUC (0.649 to 0.636). But I checked whether it also spread
the queue across more clients, which was the reason I built it, and it did the **opposite**: the
largest single client's share of the top 50 rose from 62% to **78%**, and the number of distinct
clients in the top 50 fell from 5 to **3**. Normalising a page against its own portfolio makes
the most internally-anomalous pages rise, and those cluster inside whichever client has the most
volatile portfolio.

So the honest reading is that this variant is the best *ranker* of the three and the worst
*distributor*. The concentration problem flagged in ML-07 is still unsolved, and now has a second
measurement against it: it needs an explicit per-client cap when the queue is built in ML-10, not
a feature-engineering fix.

**Honest bottom line:** on a client-grouped split, a learned model beats a transparent
hand-written rule on this problem by a modest margin. The signal is real, it is weak, and the
main deliverable of this week is knowing the difference between 0.649 and the 0.776 I would have
reported with a careless split.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.